In [ ]:
import hierarchical_binary_quantization.hbq as hbq
import hierarchical_binary_quantization.example_autoencoder_with_rope as eae
import lpips
import matplotlib.pyplot as plt
import math, random
import os
import time
import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TF
import hierarchical_binary_quantization.misc.checkpoint_helpers as ch
from datetime import datetime
from hierarchical_binary_quantization.example_autoencoder_with_rope import ExampleQuantizingAutoencoderWithRope
from hierarchical_binary_quantization.misc.checkpoint_helpers import save_checkpoint,load_checkpoint

import hierarchical_binary_quantization.example_narrow_reciptive_field_autoencoder as enrfa

from pathlib import Path
from PIL import Image
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import Dataset
from tqdm.auto import tqdm
from datetime import timedelta

dataset='fantasy'
batch_size=4
target_size=256
training_device = "cpu"

#cp_path = 'checkpoints/good/2026-08-04_02:00:00finetune_256_5_bd_96_ld_32_qd_8.pth'
#cp_path = 'checkpoints/ExampleQuantizingAutoencoderWithRope_2026-09-01_22-34-06_16x16_256_5_bd_96_ld_32_qd_16.pth'
#cp_path = 'checkpoints/good/ExampleQuantizingAutoencoderWithRope_2026-09-03_06-23-04_32x32_256_4_bd_96_ld_32_qd_16.pth'

cp_path = None
if False:
    cfgdict = ch.check_checkpoint(cp_path)['config']
    cfg = eae.HBQAutoencoderConfig(**cfgdict)
    print(cfg)
else:
    cfg = eae.HBQAutoencoderConfig(
        in_channels=3, 
        base_dim=96, 
        channel_multipliers=(1, 2, 3, 4), 
        res_blocks=(1, 2, 3, 3), 
        latent_dim=32, quant_dim=8, n_rounds=4
    )

cfg_32x32 = eae.HBQAutoencoderConfig(
    in_channels=3,
    base_dim=96,
    channel_multipliers=(1, 2, 3, 4),
    res_blocks=(1, 2, 3, 3),
    latent_dim=32,
    quant_dim=16,
    n_rounds=4,
    bottleneck_heads=8,
    bottleneck_layers=2,
    use_mae_aux=True,
    mae_mask_ratio=0.5,
    mae_heads=4,
    mae_depth=2,
    protected_channel_fraction=0.5,
)

cfg_16x16 = eae.HBQAutoencoderConfig(
    in_channels=3,
    base_dim=96,
    channel_multipliers=(1, 2, 3, 4, 4),
    res_blocks=(1, 2, 3, 3, 3),
    latent_dim=32,
    quant_dim=16,
    n_rounds=4,
    bottleneck_heads=8,
    bottleneck_layers=2,
    use_mae_aux=True,
    mae_mask_ratio=0.5,
    mae_heads=4,
    mae_depth=2,
    protected_channel_fraction=0.5,
)


cp_path = None
if cp_path:
    cfgdict = ch.check_checkpoint(cp_path)['config']
    cfg_l = eae.LocalAutoencoderConfig(**cfgdict)
    print(cfg_l)
else:
    """
    class LocalAutoencoderConfig:
    in_channels: int = 3
    stem_dim: int = 64
    stem_blocks: int = 2
    patch_size: int = 16
    body_dim: int = 256
    body: tuple = ("res", "attn", "res", "attn", "res", "attn", "res", "attn")
    attn_block: int = 4        # neighborhood-attention query block size, in TOKENS
    attn_halo: int = 1         # neighborhood-attention halo, in TOKENS
    attn_heads: int = 4
    latent_dim: int = 32
    quant_dim: int = 8
    n_rounds: int = 4
    refine_blocks: int = 2     # post-de-patchify local cleanup at full resolution
    use_mae_aux: bool = True
    mae_mask_ratio: float = 0.5
    mae_heads: int = 2
    mae_depth: int = 2
    """
    cfg_l8 = enrfa.LocalAutoencoderConfig( patch_size=8, body_dim=256, quant_dim=16, n_rounds=4, body = ("res", "attn", "res", "attn"))
    cfg_l16 = enrfa.LocalAutoencoderConfig( patch_size=16, body_dim=256, quant_dim=16, n_rounds=4, body = ("res", "attn", "res", "attn"))
    
cfg = cfg_l8


def get_autoencoder(cfg):
    global batch_size
    batch_size = 2
    if isinstance(cfg,enrfa.LocalAutoencoderConfig):
        autoencoder = enrfa.LocalQuantizingAutoencoder(cfg)
    else:
        autoencoder = eae.ExampleQuantizingAutoencoderWithRope(cfg)
    return autoencoder


In [ ]:
autoencoder = enrfa.LocalQuantizingAutoencoder(enrfa.LocalAutoencoderConfig())

import hierarchical_binary_quantization.example_narrow_reciptive_field_autoencoder as enrfa
enrfa.receptive_field_pixels(enrfa.LocalAutoencoderConfig())

In [ ]:
from torchview import draw_graph
x = torch.randn(1, 3, 32, 32)
y, aux = autoencoder(x)
#aew = AutoencoderWrapper(autoencoder)
#aew.forward(x)

# torchviz ... (y, params=dict(aew.named_parameters())).render("simple_nn", format="png")

# device='meta' -> no memory is consumed for visualization
model_graph = draw_graph(autoencoder,                          
                        input_data = x,
                        device='cpu',
                        depth=2,
                        expand_nested=True,
                        hide_inner_tensors=True,
                        roll=True,
                        hide_module_functions=False,
)
model_graph.visual_graph


In [ ]:
1

In [ ]:

def show_model_info(model_edm):
    # Totals
    results = []
    total_params = sum(p.numel() for p in model_edm.parameters())
    trainable_params = sum(p.numel() for p in model_edm.parameters() if p.requires_grad)
    results.append(model_edm.__class__.__name__)
    results.append(f"Total parameters: {total_params:,}")
    results.append(f"Trainable parameters: {trainable_params:,} ({trainable_params/total_params*100:.2f}%)\n")
    
    # Breakdown by top-level module (first name segment)
    by_module = {}
    for name, p in model_edm.named_parameters():
        top = name.split('.')[0]
        tot = p.numel()
        by_module.setdefault(top, [0, 0])
        by_module[top][0] += tot
        if p.requires_grad:
            by_module[top][1] += tot
    
    # Print sorted breakdown
    results.append("Parameter breakdown by top-level module:")
    for mod, (tot, train) in sorted(by_module.items(), key=lambda x: x[1][0], reverse=True):
        pct = train / tot * 100 if tot else 0.0
        results.append(f"{mod:35} total: {tot:12,}   trainable: {train:12,}   trainable%: {pct:6.2f}")
    
    # Show largest individual parameter tensors for quick inspection
    results.append("\nTop 20 largest parameter tensors:")
    largest = sorted(model_edm.named_parameters(), key=lambda x: x[1].numel(), reverse=True)[:20]
    for name, p in largest:
        results.append(f"{name:60} shape: {tuple(p.shape)} params: {p.numel():12,}  {'train' if p.requires_grad else 'frozen'}")
    return "\n".join(results)

In [ ]:
print(show_model_info))

In [ ]:
print(show_model_info(
    eae.ExampleQuantizingAutoencoderWithRope(
        eae.HBQAutoencoderConfig(
    in_channels=3, base_dim=64, channel_multipliers=(1,2,4,4,4), res_blocks=(1,1,1,1,2),
    latent_dim=64, quant_dim=16, n_rounds=4, bottleneck_heads=8, bottleneck_layers=2,
    protected_channel_fraction=0.5, use_mae_aux=True, mae_mask_ratio=0.5, mae_heads=4, mae_depth=2,
        )
    )
))

In [ ]:
print(show_model_info(
    eae.ExampleQuantizingAutoencoderWithRope(
        eae.HBQAutoencoderConfig(
            in_channels=3,
            base_dim=96,
            channel_multipliers=(1, 2, 3, 4, 5),
            res_blocks=(1, 1, 1, 1, 1),
            latent_dim=32,
            quant_dim=16,
            n_rounds=4,
            bottleneck_heads=8,
            bottleneck_layers=2,
            use_mae_aux=True,
            mae_mask_ratio=0.5,
            mae_heads=4,
            mae_depth=2,
            protected_channel_fraction=0.5,
        )
    )
))

In [ ]:
print(show_model_info(
    eae.ExampleQuantizingAutoencoderWithRope(
        eae.HBQAutoencoderConfig(
            in_channels=3,
            base_dim=64,
            channel_multipliers=(1, 2, 3, 4, 4),
            res_blocks=(1, 1, 1, 1, 1),
            latent_dim=32,
            quant_dim=16,
            n_rounds=4,
            bottleneck_heads=8,
            bottleneck_layers=1,
            use_mae_aux=True,
            mae_mask_ratio=0.5,
            mae_heads=4,
            mae_depth=2,
            protected_channel_fraction=0.5,
        )
    )
))

In [ ]:
print(show_model_info(
    enrfa.LocalQuantizingAutoencoder(
        enrfa.LocalAutoencoderConfig( patch_size=16, body_dim=256, quant_dim=16, n_rounds=4, latent_dim=32, body = ("res", "attn", "res"), depatchify_overlap=1))
    )
)

In [ ]:
cfg = enrfa.LocalAutoencoderConfig(in_channels=3, stem_dim=64, stem_blocks=2, patch_size=8, body_dim=256, body=('res', 'attn', 'res', 'attn'), attn_block=4, attn_halo=1, attn_heads=4, latent_dim=32, quant_dim=16, n_rounds=4, refine_blocks=2, use_mae_aux=True, mae_mask_ratio=0.5, mae_heads=2, mae_depth=2)
cfg

In [ ]:
tag = f"ps{cfg.patch_size},qd{cfg.quant_dim},ld{cfg.latent_dim},body:{",".join(cfg.body)}"